In [1]:
import yfinance as yf
import pandas as pd
import sqlalchemy 
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

In [2]:
load_dotenv("../../.env")

mysql_host = os.environ.get("MYSQL_HOST")
mysql_user = os.environ.get("MYSQL_USER")
mysql_password = os.environ.get("MYSQL_PASSWORD")
mysql_database = os.environ.get("MYSQL_DATABASE")


#the f goes in front to embed variables
engine = sqlalchemy.create_engine(f"mysql+mysqlconnector://{mysql_user}:{mysql_password}@{mysql_host}/{mysql_database}")

with engine.connect() as conn:
    print("Connection successful")

# os.getcwd()

Connection successful


In [84]:
tickers = yf.Tickers('EQIX DLR IRM')

raw_data = yf.download("EQIX DLR IRM", period = '1y')

raw_data.head()


# The table intially started in a multilayered column format. The following code adjusts it to row format to match the format of the MYSQL databases
raw_pivot = raw_data.stack([0,1])
raw_pivot = raw_pivot.unstack(1)
raw_pivot = raw_pivot.reset_index([0,1])
raw_pivot.columns.name = None

#renaming the columns to match the sql schema
raw_pivot = raw_pivot.rename(columns={
    "Close": "adjusted_close",
    "High": "high",
    "Low": "low",
    "Open": "open",
    "Volume": "volume",
    "Date": "date",
    "Ticker": "ticker"
})
raw_pivot["volume"] = raw_pivot["volume"].astype(int)



print(raw_pivot.head)
raw_pivot.dtypes

[*********************100%***********************]  3 of 3 completed

<bound method NDFrame.head of           date ticker  adjusted_close         high          low         open  \
0   2025-06-06    DLR      171.828217   172.770727   171.274378   171.964248   
1   2025-06-06   EQIX      894.095032   896.989229   889.030231   894.476373   
2   2025-06-06    IRM       98.748947   100.285110    98.478430    98.729628   
3   2025-06-09    DLR      171.672760   173.363453   170.701101   172.411231   
4   2025-06-09   EQIX      887.299683   894.936004   886.155717   891.611648   
..         ...    ...             ...          ...          ...          ...   
748 2026-06-04   EQIX     1089.150024  1091.969971  1061.339966  1079.930054   
749 2026-06-04    IRM      130.250000   130.279999   125.599998   129.250000   
750 2026-06-05    DLR      186.789993   188.824997   186.080002   187.190002   
751 2026-06-05   EQIX     1080.949951  1093.000000  1076.800049  1082.150024   
752 2026-06-05    IRM      124.660004   129.320404   124.180000   129.039993   

      vol


X:\Temp\ipykernel_15336\905247997.py:9: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  raw_pivot = raw_data.stack([0,1])


date              datetime64[ns]
ticker                    object
adjusted_close           float64
high                     float64
low                      float64
open                     float64
volume                     int64
dtype: object

In [83]:
#This is writing the company info to the companies table

company_df = pd.DataFrame({ 
    "ticker": ['EQIX', 'IRM', 'DLR'],
    "company_name": ["Equinix", "Iron Mountain", 'Digital Reality Trust'],
    "sector": ["REIT", "REIT", "REIT"],
    "exchange": ["NYSE", "NYSE", "NYSE"]
})

company_df.to_sql(name = 'companies', con = engine, if_exists = 'append', index = False)


IntegrityError: (mysql.connector.errors.IntegrityError) 1062 (23000): Duplicate entry 'EQIX' for key 'companies.PRIMARY'
[SQL: INSERT INTO companies (ticker, company_name, sector, exchange) VALUES (%(ticker)s, %(company_name)s, %(sector)s, %(exchange)s)]
[parameters: [{'ticker': 'EQIX', 'company_name': 'Equinix', 'sector': 'REIT', 'exchange': 'NYSE'}, {'ticker': 'IRM', 'company_name': 'Iron Mountain', 'sector': 'REIT', 'exchange': 'NYSE'}, {'ticker': 'DLR', 'company_name': 'Digital Reality Trust', 'sector': 'REIT', 'exchange': 'NYSE'}]]
(Background on this error at: https://sqlalche.me/e/20/gkpj)

In [85]:
raw_pivot.to_sql(name = 'daily_prices', con = engine, if_exists = 'append', index = False)

753